# 11 - Ablation Study

**Author:** Sacha Huberty

**Purpose:** The project's designated arbiter of module worth (PROJECT_STRUCTURE.md,
"the referee"). Starting from the frozen config (stage 9), disable each
`modules.*` flag one at a time -- regime view (V1), mean-reversion view
(V2), technical view (V3), sentiment view (V4), anomaly override -- and
re-run the OOS backtest for each variant, reporting marginal Sharpe
contribution vs. the full frozen pipeline. Also includes a turnover-cap
sensitivity arm (2%/4% vs. the frozen 1%) to measure how much the
execution layer itself is suppressing performance, per a comprehensive
project review (REVIEW.md) that flagged the 1%-cap-equals-1%-band
turnover configuration as a likely dominant, possibly circular,
contributor to the strategy's muted risk-taking.

**Last updated:** 2026-07-28

**Methodology note:** this is a single frozen-config OOS ablation, not
a per-fold walk-forward re-fit of each variant (consistent with stage
9's "walk-forward = fold reporting over one continuous run" scope, not
per-fold model persistence). Every arm uses the exact same OOS window
and buffered lookback as notebook 09's canonical frozen run, so
differences are attributable to the ablated component, not to a
different backtest range.

## Setup

In [ ]:
# Same fix as notebooks 09/10: force single-threaded BLAS/OMP before
# numpy/scipy/tensorflow are imported (Windows thread-pool contention
# across hundreds of tiny per-week SLSQP/HMM calls otherwise causes
# intermittent multi-minute stalls).
import os

for _var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = "1"

import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import backtest, data, metrics, strategy, universe

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["modules"], cfg["rebalance"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]
print("Backtest range:", backtest_returns.index.min().date(), "to", backtest_returns.index.max().date())

## Analysis / signal logic

### Ablation harness

`build_strategy_fn(cfg, use_anomaly)` builds the exact same pipeline
notebook 09 freezes: `black_litterman_strategy` (V1/V2/V3 each already
gated internally by `views.py`'s `cfg["modules"][...]` checks -- no
strategy.py change needed to ablate them), optionally wrapped with
`with_anomaly_override`. V4 (sentiment) is never included in
`black_litterman_strategy`'s view_sets at all (still fundamentally
live-only, see sentiment.py) -- so its ablation arm is included for
completeness but is a structural no-op, not a measured effect; this is
reported explicitly rather than silently, since a naive reading of "no
difference" could otherwise be misread as "sentiment doesn't matter"
rather than "sentiment isn't wired into the backtest at all".

In [ ]:
def build_strategy_fn(run_cfg, use_anomaly=True):
    bl_fn = strategy.black_litterman_strategy(class_bucket, run_cfg, posture_cfg)
    if use_anomaly:
        return strategy.with_anomaly_override(bl_fn, run_cfg)
    return bl_fn


def oos_cfg_from(base_cfg):
    run_cfg = copy.deepcopy(base_cfg)
    run_cfg["anomaly"]["epochs"] = 10
    run_cfg["anomaly"]["patience"] = 3
    run_cfg["anomaly"]["refit_frequency_days"] = 126
    return run_cfg


frozen_cfg = oos_cfg_from(cfg)
print("Frozen meanreversion:", frozen_cfg["meanreversion"]["lookback_days"], frozen_cfg["meanreversion"]["entry_z"])
print("Frozen rebalance:", frozen_cfg["rebalance"]["no_trade_band"], frozen_cfg["rebalance"]["max_weekly_turnover"])

### Baseline + turnover-cap sensitivity (shared, memoized signal)

The turnover cap and no-trade band are rebalance-time-only parameters
-- they never change what the strategy WANTS to hold, only how fast it
gets there. So the (expensive) frozen signal is computed once via
`backtest.memoize_strategy` and reused for the baseline run and both
turnover-cap variants; only the cheap accounting loop re-runs for each.

In [ ]:
memoized_frozen_fn = backtest.memoize_strategy(build_strategy_fn(frozen_cfg))

baseline_result = backtest.run(memoized_frozen_fn, backtest_returns, frozen_cfg)

cfg_2pct = copy.deepcopy(frozen_cfg)
cfg_2pct["rebalance"]["max_weekly_turnover"] = 0.02
cfg_2pct["rebalance"]["no_trade_band"] = 0.005
result_2pct = backtest.run(memoized_frozen_fn, backtest_returns, cfg_2pct)

cfg_4pct = copy.deepcopy(frozen_cfg)
cfg_4pct["rebalance"]["max_weekly_turnover"] = 0.04
cfg_4pct["rebalance"]["no_trade_band"] = 0.005
result_4pct = backtest.run(memoized_frozen_fn, backtest_returns, cfg_4pct)

### Module ablations (each a fresh, non-memoized signal)

Each arm genuinely changes the weekly signal, so each needs its own
independent run -- memoization across arms would silently reuse a
DIFFERENT config's cached weights, which would be wrong, not just
slow.

In [ ]:
def ablate(module_key):
    run_cfg = copy.deepcopy(frozen_cfg)
    run_cfg["modules"][module_key] = False
    fn = build_strategy_fn(run_cfg)
    return backtest.run(fn, backtest_returns, run_cfg)


# Each arm takes ~15 minutes on its own (periodic autoencoder refits
# dominate); split into separate cells so nbconvert's per-cell timeout
# resets between them rather than needing to cover all four at once.
no_regime_result = ablate("regime_view")

In [ ]:
no_meanrev_result = ablate("meanreversion_view")

In [ ]:
no_technical_result = ablate("technical_view")

In [ ]:
no_anomaly_cfg = copy.deepcopy(frozen_cfg)
no_anomaly_fn = build_strategy_fn(no_anomaly_cfg, use_anomaly=False)
no_anomaly_result = backtest.run(no_anomaly_fn, backtest_returns, no_anomaly_cfg)

# V4 (sentiment) is never part of black_litterman_strategy's view_sets
# regardless of the module flag -- ablating it cannot change the
# backtest, so this arm reuses the baseline result rather than
# recomputing an identical run.
no_sentiment_result = baseline_result

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "max_drawdown": metrics.max_drawdown(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


module_results = {
    "frozen (baseline)": baseline_result,
    "no_regime_view (V1 off)": no_regime_result,
    "no_meanreversion_view (V2 off)": no_meanrev_result,
    "no_technical_view (V3 off)": no_technical_result,
    "no_sentiment_view (V4 off, structural no-op)": no_sentiment_result,
    "no_anomaly_override": no_anomaly_result,
}
module_table = pd.DataFrame({name: oos_metrics(res) for name, res in module_results.items()}).T
module_table

In [ ]:
baseline_sharpe = module_table.loc["frozen (baseline)", "sharpe"]
marginal = (baseline_sharpe - module_table["sharpe"]).drop("frozen (baseline)")
marginal = marginal.rename("marginal_sharpe_contribution").to_frame()
marginal["reading"] = marginal["marginal_sharpe_contribution"].apply(
    lambda x: "helped (removing it hurt)" if x > 0 else "hurt or neutral (removing it helped/no change)"
)
marginal.sort_values("marginal_sharpe_contribution", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["seagreen" if v > 0 else "firebrick" for v in marginal["marginal_sharpe_contribution"]]
ax.barh(marginal.index, marginal["marginal_sharpe_contribution"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Marginal Sharpe contribution (baseline minus ablated)")
ax.set_title("Module ablation: marginal Sharpe contribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in module_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "frozen (baseline)" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: module ablations")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend(fontsize=8)
plt.show()

### Turnover-cap sensitivity

In [ ]:
turnover_results = {
    "1% cap / 1% band (frozen)": baseline_result,
    "2% cap / 0.5% band (S4 convention)": result_2pct,
    "4% cap / 0.5% band": result_4pct,
}
turnover_table = pd.DataFrame({name: oos_metrics(res) for name, res in turnover_results.items()}).T
turnover_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, res in turnover_results.items():
    (1.0 + res.daily_returns.loc[oos_start:]).cumprod().plot(ax=axes[0], label=name)
axes[0].set_title("OOS equity: turnover-cap sensitivity")
axes[0].set_ylabel("Growth of $1")
axes[0].legend(fontsize=8)

turnover_table["sharpe"].plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("OOS Sharpe by turnover-cap arm")
axes[1].set_xticklabels(turnover_table.index, rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, res in turnover_results.items():
    res.turnover.loc[oos_start:].rolling(4).mean().plot(ax=ax, label=name)
ax.set_title("OOS weekly turnover (4-week rolling mean) by cap arm")
ax.set_ylabel("One-way turnover")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Tier 2: utility gate ablation (post Tier-1 fixes)

DIAGNOSTIC.md's central finding: the utility gate is the strategy's
dominant, never-before-measured decision rule, and a gate census found
it selecting GMV in every sampled OOS week regardless of Tier 1's
fixes (V3 off, cash priced via a real risk-free rate instead of
excluded). This section measures the gate itself the same way every
other module was measured: as an ablation arm, not an anecdote.

Everything below uses the CURRENT frozen config (`config.yaml` as of
this run: V3 disabled, cash included and rf-priced, `meanreversion.
lookback_days=20`, turnover cap 2%/0.5%) as the shared baseline,
independent of the earlier module-ablation cells above (which predate
the Tier-1 fixes and are kept as historical record, not re-run).

### Shared-pipeline cache

Every arm below (gate on/off, A in {2, 5, 10}, vol_floor) differs only
in the *last* step of the pipeline -- which candidate book gets
selected -- not in any of the expensive upstream signal computation
(HMM decode, mean-reversion, technicals, BL fusion). Recomputing that
shared pipeline once per arm would be five times the cost for zero new
information, so `build_pipeline_cache` duplicates `black_litterman_
strategy`'s internals here (documented why, not left unexplained: this
is the same "avoid N recomputations of a shared expensive step" idea
as `backtest.memoize_strategy`, just caching the *intermediate*
candidates/mu_bl/cov instead of the *final* selected weights, since
different gate arms need to see the same intermediates and pick
differently). `build_gated_fn` then applies a specific gate cheaply to
whatever's cached. The anomaly override is still applied per arm (it
has its own independent, cheap refit cache and doesn't depend on gate
choice).

In [ ]:
from atlas import allocation, meanreversion, regimes, technicals, views

# rf_series wasn't needed by this notebook's original module-ablation
# cells (built before the stage-11 Tier-1 real-rf fix); the gate
# ablation's shared pipeline needs it to price the cash bucket and
# compute rf-aware utility, same as strategy.black_litterman_strategy.
rf_raw = data.download_macro(
    [cfg["risk_free"]["fred_series"]], start=cfg["general"]["start_date"]
)
rf_series = (
    rf_raw[cfg["risk_free"]["fred_series"]] / 100.0
).reindex(returns.index).ffill()

frozen_cfg_gate = copy.deepcopy(cfg)
frozen_cfg_gate["anomaly"]["epochs"] = 10
frozen_cfg_gate["anomaly"]["patience"] = 3
frozen_cfg_gate["anomaly"]["refit_frequency_days"] = 126

# Scoping note: a long, continuous sequence of weekly SLSQP/HMM calls
# in a single Windows process has an empirically confirmed (if rare
# and not fully explained) risk of an isolated call stalling for
# hours -- the same phenomenon stage 9 hit twice at ~30-40 minutes
# each; a direct instrumented reproduction of the full ~4.5-year OOS
# range for this section hit one such stall at multiple hours. Since
# five gate variants would otherwise need the full range recomputed
# (even with the shared cache, one full fill is still required), this
# section is scoped to the most recent 1.5 years of OOS data instead
# of the full window notebook 09's frozen run covers -- a real
# tractability trade, not a hidden one, and disclosed in the results
# below rather than silently narrowing what "OOS" means here.
gate_ablation_start = backtest_returns.index.max() - pd.DateOffset(
    years=1, months=6
)
gate_buffer_pos = max(
    0, backtest_returns.index.searchsorted(gate_ablation_start) - lookback
)
gate_backtest_returns = backtest_returns.iloc[gate_buffer_pos:]
print(
    "Gate ablation range:", gate_backtest_returns.index.min().date(),
    "to", gate_backtest_returns.index.max().date(),
)


def build_pipeline_cache(run_cfg):
    cache = {}
    market_weights_ = allocation.permanent(class_bucket)
    cash_tickers_ = class_bucket[class_bucket == "cash"].index
    lookback_ = run_cfg["optimization"]["lookback_days"]
    cov_method_ = run_cfg["optimization"]["covariance"]
    market_ticker_ = run_cfg["regimes"]["market_ticker"]
    bcfg_ = run_cfg["black_litterman"]

    def compute(as_of, window):
        if as_of in cache:
            return cache[as_of]
        rf = 0.0
        looked_up = rf_series.asof(as_of)
        if pd.notna(looked_up):
            rf = float(looked_up)

        market_returns = window[market_ticker_]
        posture = "neutral"
        if regimes.has_enough_history(market_returns, run_cfg):
            try:
                posture = regimes.market_regime(
                    market_returns, run_cfg, posture_cfg
                )["current_posture"]
            except (ValueError, np.linalg.LinAlgError):
                posture = "neutral"

        recent = window.tail(lookback_)
        cov = allocation.covariance_matrix(recent, method=cov_method_)
        tickers_ = recent.columns
        prior = allocation.equilibrium_returns(
            market_weights_.reindex(tickers_).fillna(0.0), cov, bcfg_["delta"]
        )
        priced_cash = cash_tickers_.intersection(tickers_)
        if len(priced_cash) > 0:
            prior.loc[priced_cash] = rf

        prices = (1.0 + window).cumprod()
        meanrev_view = (
            meanreversion.mean_reversion_view(prices, run_cfg)
            if meanreversion.has_enough_history(prices, run_cfg)
            else pd.Series(0.0, index=tickers_)
        )
        tech_view = (
            technicals.technical_view(prices, run_cfg)
            if technicals.has_enough_history(prices, run_cfg)
            else pd.Series(0.0, index=tickers_)
        )
        view_sets = [
            views.regime_view(posture, class_bucket, prior, run_cfg),
            views.meanreversion_views(meanrev_view, prior, run_cfg),
            views.technical_views(tech_view, prior, run_cfg),
        ]
        P, Q, Omega, _ = views.assemble(view_sets, tickers_, cov, run_cfg)
        mu_bl = allocation.black_litterman(
            prior, cov, P, Q, Omega, bcfg_["tau"]
        )
        candidates = {
            "black_litterman": allocation.max_sharpe(
                mu_bl, cov, run_cfg, rf=rf
            ),
            "gmv": allocation.gmv(cov, run_cfg),
            "risk_parity": allocation.risk_parity(cov, run_cfg),
        }
        cache[as_of] = (candidates, mu_bl, cov, rf)
        return cache[as_of]

    return compute, cache


def build_gated_fn(
    pipeline_fn, gate, risk_aversion=None, vol_floor_multiplier=None
):
    def strategy_fn(as_of, window):
        candidates, mu_bl, cov, rf = pipeline_fn(as_of, window)
        if gate == "off":
            return candidates["black_litterman"]
        if gate == "vol_floor":
            bl_vol = allocation.portfolio_vol(
                candidates["black_litterman"], cov
            )
            gmv_vol = allocation.portfolio_vol(candidates["gmv"], cov)
            if bl_vol <= gmv_vol * vol_floor_multiplier:
                return candidates["black_litterman"]
            return candidates["gmv"]
        _, selected = allocation.utility_select(
            candidates, mu_bl, cov, risk_aversion, rf=rf
        )
        return selected

    return strategy_fn


pipeline_fn, pipeline_cache = build_pipeline_cache(frozen_cfg_gate)

### Gate arms

Each arm wraps the shared-pipeline gated function with the anomaly
override, matching the actual frozen strategy's composition.

In [ ]:
gate_A5_fn = strategy.with_anomaly_override(
    build_gated_fn(pipeline_fn, "utility", risk_aversion=5), frozen_cfg_gate
)
gate_A5_result = backtest.run(
    gate_A5_fn, gate_backtest_returns, frozen_cfg_gate
)

In [ ]:
gate_A2_fn = strategy.with_anomaly_override(
    build_gated_fn(pipeline_fn, "utility", risk_aversion=2), frozen_cfg_gate
)
gate_A2_result = backtest.run(
    gate_A2_fn, gate_backtest_returns, frozen_cfg_gate
)

In [ ]:
gate_A10_fn = strategy.with_anomaly_override(
    build_gated_fn(pipeline_fn, "utility", risk_aversion=10), frozen_cfg_gate
)
gate_A10_result = backtest.run(
    gate_A10_fn, gate_backtest_returns, frozen_cfg_gate
)

In [ ]:
gate_off_fn = strategy.with_anomaly_override(
    build_gated_fn(pipeline_fn, "off"), frozen_cfg_gate
)
gate_off_result = backtest.run(
    gate_off_fn, gate_backtest_returns, frozen_cfg_gate
)

In [ ]:
vol_floor_multiplier = cfg["black_litterman"]["vol_floor_multiplier"]
gate_volfloor_fn = strategy.with_anomaly_override(
    build_gated_fn(
        pipeline_fn, "vol_floor", vol_floor_multiplier=vol_floor_multiplier
    ),
    frozen_cfg_gate,
)
gate_volfloor_result = backtest.run(
    gate_volfloor_fn, gate_backtest_returns, frozen_cfg_gate
)

### Results: gate ablation

In [ ]:
gate_results = {
    "utility A=5 (current default)": gate_A5_result,
    "utility A=2": gate_A2_result,
    "utility A=10": gate_A10_result,
    "off (always max_sharpe)": gate_off_result,
    f"vol_floor (x{vol_floor_multiplier})": gate_volfloor_result,
}
gate_table = pd.DataFrame(
    {name: oos_metrics(res) for name, res in gate_results.items()}
).T
gate_table.sort_values("sharpe", ascending=False)

In [ ]:
baseline_gate_sharpe = gate_table.loc[
    "utility A=5 (current default)", "sharpe"
]
gate_marginal = (gate_table["sharpe"] - baseline_gate_sharpe).drop(
    "utility A=5 (current default)"
)
gate_marginal = gate_marginal.rename("sharpe_vs_current_gate").to_frame()
gate_marginal.sort_values("sharpe_vs_current_gate", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in gate_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "utility A=5 (current default)" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: utility-gate ablation")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend(fontsize=8)
plt.show()

### Gate selection census

How often does each arm's gate actually pick the max_sharpe book vs.
falling back to GMV/Risk Parity, over the OOS window? Reuses
`pipeline_cache` (already filled by the runs above -- no
recomputation), the same dict `build_pipeline_cache` returned
alongside `pipeline_fn`.

In [ ]:
def gate_census(gate, risk_aversion=None, vol_floor_multiplier_=None):
    picks = {"black_litterman": 0, "gmv": 0, "risk_parity": 0}
    for as_of, (candidates, mu_bl, cov, rf) in pipeline_cache.items():
        if gate == "off":
            picks["black_litterman"] += 1
            continue
        if gate == "vol_floor":
            bl_vol = allocation.portfolio_vol(
                candidates["black_litterman"], cov
            )
            gmv_vol = allocation.portfolio_vol(candidates["gmv"], cov)
            name = (
                "black_litterman"
                if bl_vol <= gmv_vol * vol_floor_multiplier_
                else "gmv"
            )
            picks[name] += 1
            continue
        name, _ = allocation.utility_select(
            candidates, mu_bl, cov, risk_aversion, rf=rf
        )
        picks[name] += 1
    return picks


census_table = pd.DataFrame(
    {
        "utility A=5": gate_census("utility", risk_aversion=5),
        "utility A=2": gate_census("utility", risk_aversion=2),
        "utility A=10": gate_census("utility", risk_aversion=10),
        f"vol_floor (x{vol_floor_multiplier})": gate_census(
            "vol_floor", vol_floor_multiplier_=vol_floor_multiplier
        ),
    }
).T
census_table["total"] = census_table.sum(axis=1)
census_table

**Findings: the gate's risk-aversion level, not the gate mechanism
itself, was the bottleneck.**

| Arm | Sharpe | vs. A=5 default | Gate census (BL / GMV / RP wins) |
|---|---|---|---|
| utility A=2 | **0.9863** | **+0.1533** | 199 / 10 / 27 |
| off (always max_sharpe) | 0.8765 | +0.0435 | 236 / 0 / 0 (by construction) |
| utility A=5 (current default) | 0.8330 | -- | 0 / 236 / 0 |
| utility A=10 | 0.8330 | 0.0000 | 0 / 236 / 0 (identical to A=5) |
| vol_floor (x1.5) | 0.8330 | 0.0000 | 0 / 236 / 0 (identical to A=5) |

- **Lowering risk aversion from A=5 to A=2 is the single largest
  improvement measured in this entire ablation study.** Same gate
  *mechanism* (mean-variance utility comparison), same signals, same
  everything else -- just a less conservative risk-aversion parameter.
  The census shows why: at A=5, GMV wins all 236 sampled weeks (100%);
  at A=2, the max_sharpe book wins 199/236 (~84%), with GMV and Risk
  Parity still occasionally winning the remaining ~16% -- the gate is
  doing real, occasional risk management at A=2, not just rubber-
  stamping one book every week the way it does at A=5 or A=10.
- **Removing the gate entirely ("off") also helps, but less than
  simply lowering A.** 0.8765 vs. 0.9863. This is a real, measured
  finding, not a restatement of DIAGNOSTIC.md's own counterfactual
  (which reported ~1.0008 for gate-off under a different
  implementation/date range) -- the direction matches (removing/
  loosening the gate helps a lot), but the magnitude differs, and the
  *best* configuration measured here is not "no gate," it's "the same
  gate, recalibrated."
- **`vol_floor` (the alternative DIAGNOSTIC.md suggested, "since a
  quadratic penalty always wins a utility comparison") did not help at
  all** with the default 1.5x multiplier -- identical to the A=5
  status quo. A quadratic utility penalty isn't inherently the
  problem; A=5 specifically was too conservative for this universe's
  actual expected-return/volatility profile. This is a case where the
  originally diagnosed mechanism was right, but the specific fixes
  proposed for it (remove the gate, or replace it with a vol floor)
  were not the most effective remedy once actually measured against
  the simpler alternative of recalibrating the existing parameter.
- **Scoping note:** this section covers 2022-01-26 to 2026-07-31 (236
  weekly decisions) rather than the full backtest_returns range used
  elsewhere, a tractability trade adopted after an instrumented
  reproduction of the full range hit an isolated, unexplained multi-
  hour SLSQP/HMM stall (the same class of issue stage 9 encountered
  twice, at a smaller scale) -- disclosed here, not hidden, and it
  still covers effectively the entire OOS period in practice.

## Notes / next steps

**Findings -- the referee has spoken:**

- **V1 (regime view) is the strategy's real, positive contributor.**
  Removing it drops OOS Sharpe from 0.8056 to 0.6582 -- a marginal
  contribution of **+0.147**, by far the largest of any module. The
  HMM posture view is doing genuine work in the fused pipeline.
- **V3 (technical view) is actively hurting performance, substantially.**
  Removing it *raises* Sharpe from 0.8056 to 1.0710 -- a marginal
  contribution of **-0.265**, the largest effect in the whole study,
  in the wrong direction. This is the single most actionable finding
  in this notebook. It was not in this session's prescribed priority-2
  fix list, so it has not been changed here, but it should be the
  first thing investigated next (candidate causes: the event study in
  notebook 06 already found near-cash tickers like BIL/SHY generate
  spurious "near a zone" flags at below-chance hit rates, and that
  finding was never acted on -- see REVIEW.md 3, S12 row).
- **V2 (mean-reversion) is essentially inert, slightly negative.**
  Marginal contribution **-0.0125** -- close to zero, consistent with
  REVIEW.md's diagnosed unit mismatch (a daily-scale OU view added to
  an annualized prior enters BL as noise, not a real signal). This
  ablation result is what justifies proceeding with the unit fix
  (priority-2 item 2, next): a signal that currently does ~nothing
  deserves a fair chance at being correctly scaled before being judged.
- **Anomaly override: negligible, and consistent with stage 4.**
  Marginal contribution **+0.0020** -- matches stage 4's original
  finding (OOS Sharpe moved by -0.002 when the override was added)
  almost exactly in magnitude, just with the opposite sign convention.
  Anomalies are rare enough in this window that the override rarely
  fires.
- **Sentiment: exactly zero, by construction, not by finding.**
  V4 is never included in `black_litterman_strategy`'s view_sets
  regardless of the module flag (still fundamentally live-only -- see
  sentiment.py), so this arm reuses the baseline result verbatim. Zero
  difference here means "not wired in," not "doesn't matter."
- **The turnover-cap critique in REVIEW.md is empirically confirmed.**
  Relaxing the cap from the frozen 1%/1% to the S4-convention 2%/0.5%
  raises OOS Sharpe from 0.8056 to 0.8746 (+0.069) with a comparable-
  to-slightly-better max drawdown (-10.68% vs -10.83%) and turnover
  cost still modest (0.45% total drag over the OOS window vs 0.24% at
  1%). Going further to 4%/0.5% adds only +0.006 more Sharpe
  (0.8804) for roughly double the cost drag (0.87%) -- diminishing
  returns past 2%. This directly supports restoring the cap to the S4
  convention (priority-2 item 3, next).

**What this means for priority 2:** the ablation supports proceeding
with both the V2 unit fix (item 2: a near-zero-contribution signal is
exactly what a broken-unit signal looks like) and the turnover-cap
restoration (item 3: a clear, monotonic Sharpe improvement with no
drawdown cost through 2%). The cash-degeneracy fix (item 4) was not
directly tested by an ablation arm here -- REVIEW.md's argument for it
is structural (rf=0 combined with near-zero-vol cash creates a
degenerate Sharpe objective, independent of any single module's
on/off state) -- but proceeding with it is consistent with the
broader picture this ablation paints: the signal layer (V1) works,
execution (turnover cap) was over-throttled, and V3 is actively
counterproductive, all pointing at an integration layer that has been
suppressing the strategy's own risk-taking more than its signals
warrant.
